# NPPE-1 Multilingual Sentiment Analysis
**Model:** `google/gemma-3-1b-it` &nbsp;|&nbsp; **Technique:** QLoRA Fine-Tuning &nbsp;|&nbsp; **Metric:** Macro F1-Score

---

### Approach
1. Load the 13-language sentiment dataset (900 train / 100 test).
2. Build language-aware instruction prompts using the Gemma chat template.
3. Apply 4-bit QLoRA fine-tuning (rank=64, alpha=128) on all linear projections.
4. Train with cosine LR schedule and paged AdamW-8bit optimizer.
5. Evaluate on a stratified validation split using Macro F1.
6. Generate test predictions via greedy decoding and produce `submission.csv`.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv


## 1 &mdash; Install Dependencies

In [3]:
import subprocess, sys

packages = ["bitsandbytes", "peft", "trl", "accelerate", "datasets"]
for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", pkg]
    )
print("All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.6 MB/s eta 0:00:00
All packages installed.


## 2 &mdash; Imports

In [4]:
import gc
import random
import warnings
from pathlib import Path

import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from trl import SFTTrainer
from datasets import Dataset

warnings.filterwarnings("ignore")

## 3 &mdash; Configuration & Reproducibility

In [5]:
class Config:
    COMPETITION_DIR = Path("/kaggle/input/competitions/nppe-dlp-2026-term-1")
    TRAIN_PATH      = COMPETITION_DIR / "train.csv"
    TEST_PATH       = COMPETITION_DIR / "test.csv"
    OUTPUT_DIR      = Path("/kaggle/working")
    MODEL_SAVE_DIR  = OUTPUT_DIR / "gemma-qlora-sentiment"

    
    MODEL_NAME = "google/gemma-3-1b-it"

    
    LORA_R       = 64
    LORA_ALPHA   = 128
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

    
    LOAD_IN_4BIT          = True
    BNB_4BIT_COMPUTE_TYPE = torch.bfloat16
    BNB_4BIT_QUANT_TYPE   = "nf4"
    USE_DOUBLE_QUANT      = True

   
    EPOCHS        = 5
    BATCH_SIZE    = 2
    GRAD_ACCUM    = 8
    LEARNING_RATE = 2e-4
    WEIGHT_DECAY  = 0.01
    WARMUP_RATIO  = 0.1
    MAX_SEQ_LEN   = 512
    VAL_SPLIT     = 0.15
    SEED          = 42

    
    MAX_NEW_TOKENS = 5


def set_seed(seed=Config.SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed()

print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/ 1e9:.1f} GB")

PyTorch: 2.9.0+cu126  |  CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


## 4 &mdash; Data Loading & Exploration

In [6]:
train_df = pd.read_csv(Config.TRAIN_PATH)
test_df  = pd.read_csv(Config.TEST_PATH)

print(f"Train shape: {train_df.shape}  |  Test shape: {test_df.shape}")
print(f"\nLabel distribution:\n{train_df['label'].value_counts()}")
print(f"\nLanguages in train:\n{train_df['language'].value_counts()}")
print(f"\nLanguages in test:\n{test_df['language'].value_counts()}")
print(f"\nMissing values — train: {train_df.isnull().sum().sum()}, test: {test_df.isnull().sum().sum()}")

train_df.head()

Train shape: (900, 4)  |  Test shape: (100, 3)

Label distribution:
label
Positive    456
Negative    444
Name: count, dtype: int64

Languages in train:
language
ta    76
hi    74
or    72
pa    72
as    71
bd    71
gu    69
ml    68
mr    67
kn    66
ur    66
bn    65
te    63
Name: count, dtype: int64

Languages in test:
language
te    14
bn    12
ur    11
kn    11
mr    10
ml     8
gu     8
as     6
bd     6
pa     5
or     5
hi     3
ta     1
Name: count, dtype: int64

Missing values — train: 0, test: 0


,ID,sentence,label,language
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml
3,304,এটি বিআইএস প্রত্যয়িত এবং নিরাপদ বলে দাবি করা হয়।,Positive,bn
4,295,6 mAh બેટરી અને લાંબા સમય સુધી ચાલતી નથી.,Negative,gu


## 5 &mdash; Prompt Engineering


In [7]:
LANGUAGE_NAMES = {
    "as": "Assamese", "bd": "Bodo",     "bn": "Bengali",
    "gu": "Gujarati", "hi": "Hindi",    "kn": "Kannada",
    "ml": "Malayalam", "mr": "Marathi",  "or": "Odia",
    "pa": "Punjabi",  "ta": "Tamil",    "te": "Telugu",
    "ur": "Urdu",
}


def build_prompt(sentence: str, language: str, label: str = None) -> str:
    
    lang_name = LANGUAGE_NAMES.get(language, "Indian")

    prompt = (
        f"<start_of_turn>user\n"
        f"Classify the sentiment of the following {lang_name} text as "
        f"exactly 'Positive' or 'Negative'. Respond with only one word.\n\n"
        f"Text: {sentence}\n"
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    if label is not None:
        prompt += f"{label}<end_of_turn>\n"
    return prompt



row = train_df.iloc[0]
print("=== Training Prompt ===")
print(build_prompt(row["sentence"], row["language"], row["label"]))
print("\n=== Inference Prompt ===")
print(build_prompt(row["sentence"], row["language"]))

=== Training Prompt ===
<start_of_turn>user
Classify the sentiment of the following Punjabi text as exactly 'Positive' or 'Negative'. Respond with only one word.

Text: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥਾਂ ਵਿੱਚ ਕੀਤਾ ਗਿਆ ਹੈ। ਤੁਹਾਨੂੰ ਹਰ ਦ੍ਰਿਸ਼ ਨੂੰ ਸਮਝਣ ਲਈ ਸਮਾਂ ਕੱਢਣਾ ਹੋਵੇਗਾ!
<end_of_turn>
<start_of_turn>model
Negative<end_of_turn>


=== Inference Prompt ===
<start_of_turn>user
Classify the sentiment of the following Punjabi text as exactly 'Positive' or 'Negative'. Respond with only one word.

Text: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥਾਂ ਵਿੱਚ ਕੀਤਾ ਗਿਆ ਹੈ। ਤੁਹਾਨੂੰ ਹਰ ਦ੍ਰਿਸ਼ ਨੂੰ ਸਮਝਣ ਲਈ ਸਮਾਂ ਕੱਢਣਾ ਹੋਵੇਗਾ!
<end_of_turn>
<start_of_turn>model



## 6 &mdash; Train / Validation Split

Stratified by label to maintain class balance in both sets.

In [8]:
skf = StratifiedKFold(
    n_splits=int(1 / Config.VAL_SPLIT), shuffle=True, random_state=Config.SEED
)
train_idx, val_idx = next(skf.split(train_df, train_df["label"]))

df_train = train_df.iloc[train_idx].reset_index(drop=True)
df_val   = train_df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(df_train)}  |  Val: {len(df_val)}")
print(f"Train labels: {df_train['label'].value_counts().to_dict()}")
print(f"Val   labels: {df_val['label'].value_counts().to_dict()}")

Train: 750  |  Val: 150
Train labels: {'Positive': 380, 'Negative': 370}
Val   labels: {'Positive': 76, 'Negative': 74}


## 7 &mdash; Dataset Preparation

In [9]:
def create_dataset(df, include_label=True):
    texts = [
        build_prompt(
            row["sentence"],
            row["language"],
            row["label"] if include_label else None,
        )
        for _, row in df.iterrows()
    ]
    return Dataset.from_dict({"text": texts})


train_dataset = create_dataset(df_train, include_label=True)
val_dataset   = create_dataset(df_val, include_label=True)

print(f"Train dataset: {len(train_dataset)}  |  Val dataset: {len(val_dataset)}")

Train dataset: 750  |  Val dataset: 150


In [10]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("Logged in to HuggingFace!")

Logged in to HuggingFace!


## 8 &mdash; Load Model (4-bit Quantized)

Gemma-3-1B-IT loaded in NF4 4-bit precision with double quantization.
This reduces GPU memory from ~4 GB to ~1.5 GB while maintaining quality.

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=Config.LOAD_IN_4BIT,
    bnb_4bit_compute_dtype=Config.BNB_4BIT_COMPUTE_TYPE,
    bnb_4bit_quant_type=Config.BNB_4BIT_QUANT_TYPE,
    bnb_4bit_use_double_quant=Config.USE_DOUBLE_QUANT,
)

tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    Config.MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=Config.BNB_4BIT_COMPUTE_TYPE,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {Config.MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded: google/gemma-3-1b-it
Vocab size: 262144


## 9 &mdash; LoRA Adapter

Low-rank adapters on all 7 linear projection layers. With rank=64 and alpha=128
the effective LoRA scaling is 2.0, giving a strong adaptation signal while keeping
trainable parameters well under 3% of the total.

In [12]:
lora_config = LoraConfig(
    r=Config.LORA_R,
    lora_alpha=Config.LORA_ALPHA,
    lora_dropout=Config.LORA_DROPOUT,
    target_modules=Config.LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total params:     {total:>12,}")
print(f"Trainable params: {trainable:>12,}")
print(f"Trainable %:      {100 * trainable / total:.2f}%")

Total params:      703,188,096
Trainable params:   52,183,040
Trainable %:      7.42%


## 10 &mdash; Training

In [13]:
training_args = TrainingArguments(
    output_dir=str(Config.MODEL_SAVE_DIR),
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    gradient_accumulation_steps=Config.GRAD_ACCUM,
    learning_rate=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,
    warmup_ratio=Config.WARMUP_RATIO,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=Config.SEED,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    dataloader_pin_memory=True,
    ddp_find_unused_parameters=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print(f"Effective batch size: {Config.BATCH_SIZE * Config.GRAD_ACCUM}")
print("Starting training...\n")

train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Runtime:    {train_result.metrics.get('train_runtime', 0):.0f}s")


trainer.save_model(str(Config.MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(Config.MODEL_SAVE_DIR))
print(f"Model saved to {Config.MODEL_SAVE_DIR}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Effective batch size: 16
Starting training...



Epoch,Training Loss,Validation Loss
1,2.301468,2.131243
2,1.756762,2.063052
3,1.155805,2.187301
4,0.816168,2.489264
5,0.579611,2.676453



Training complete!
Final loss: 1.4725
Runtime:    2015s
Model saved to /kaggle/working/gemma-qlora-sentiment


## 11 &mdash; Inference Helper

In [14]:
def predict_sentiment(sentence, language, model, tokenizer):

    prompt = build_prompt(sentence, language, label=None)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=Config.MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=Config.MAX_NEW_TOKENS,
            do_sample=False,
        )

    
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip().lower()

    if "positive" in response:
        return "Positive"
    elif "negative" in response:
        return "Negative"
    return "Positive"  

## 12 &mdash; Validation Evaluation

In [15]:
model.eval()

val_preds  = []
val_labels = df_val["label"].tolist()

for idx, row in df_val.iterrows():
    pred = predict_sentiment(row["sentence"], row["language"], model, tokenizer)
    val_preds.append(pred)
    if (idx + 1) % 25 == 0:
        print(f"  Validated {idx + 1}/{len(df_val)}...")

macro_f1 = f1_score(val_labels, val_preds, average="macro")

print(f"\n{'=' * 45}")
print(f"  VALIDATION MACRO F1-SCORE:  {macro_f1:.4f}")
print(f"{'=' * 45}")
print()
print(classification_report(val_labels, val_preds, digits=4))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Validated 25/150...
  Validated 50/150...
  Validated 75/150...
  Validated 100/150...
  Validated 125/150...
  Validated 150/150...

  VALIDATION MACRO F1-SCORE:  0.8529

              precision    recall  f1-score   support

    Negative     0.8824    0.8108    0.8451        74
    Positive     0.8293    0.8947    0.8608        76

    accuracy                         0.8533       150
   macro avg     0.8558    0.8528    0.8529       150
weighted avg     0.8555    0.8533    0.8530       150



## 13 &mdash; Per-Language F1 Breakdown

In [16]:
print(f"{'Language':<15} {'Code':<6} {'N':<5} {'F1':<8}")
print("-" * 38)

for lang in sorted(df_val["language"].unique()):
    mask = df_val["language"] == lang
    if mask.sum() > 0:
        lt = [val_labels[i] for i in range(len(val_labels)) if mask.iloc[i]]
        lp = [val_preds[i]  for i in range(len(val_preds))  if mask.iloc[i]]
        lf = f1_score(lt, lp, average="macro", zero_division=0)
        print(f"{LANGUAGE_NAMES.get(lang, lang):<15} {lang:<6} {mask.sum():<5} {lf:.4f}")

Language        Code   N     F1      
--------------------------------------
Assamese        as     11    0.8706
Bodo            bd     12    0.6571
Bengali         bn     18    0.9443
Gujarati        gu     13    0.9150
Hindi           hi     11    1.0000
Kannada         kn     14    0.8542
Malayalam       ml     12    0.8333
Marathi         mr     9     1.0000
Odia            or     14    0.4269
Punjabi         pa     8     0.7333
Tamil           ta     9     0.8615
Telugu          te     9     1.0000
Urdu            ur     10    0.8990


## 14 &mdash; Test Set Predictions

In [17]:
test_preds = []

for idx, row in test_df.iterrows():
    pred = predict_sentiment(row["sentence"], row["language"], model, tokenizer)
    test_preds.append(pred)
    if (idx + 1) % 25 == 0:
        print(f"  Predicted {idx + 1}/{len(test_df)}...")

print(f"\nPrediction distribution:")
print(pd.Series(test_preds).value_counts())

  Predicted 25/100...
  Predicted 50/100...
  Predicted 75/100...
  Predicted 100/100...

Prediction distribution:
Positive    54
Negative    46
Name: count, dtype: int64


In [18]:
pred = [1 if x == "Positive" else 0 for x in test_preds]

## 15 &mdash; Create Submission

In [22]:
submission = pd.DataFrame({"ID": test_df["ID"], "label": pred})


assert list(submission.columns) == ["ID", "label"]
assert len(submission) == len(test_df)
assert submission["label"].isin([1, 0]).all()

submission.to_csv(Config.OUTPUT_DIR / "submission.csv", index=False)

print(f"submission.csv saved  ({len(submission)} rows)")
print(f"\n{submission.head(10).to_string(index=False)}")
print(f"\n{submission['label'].value_counts()}")


submission.csv saved  (100 rows)

 ID  label
550      1
397      1
757      1
407      0
294      0
672      1
598      0
839      0
921      0
554      0

label
1    54
0    46
Name: count, dtype: int64
